In [1]:
# Fix SSL certificate issue on Windows (must run before other imports)
# Patch SSL to avoid loading corrupted Windows certificate store
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

# Alternative: use certifi bundle
import os
import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

In [2]:
# Imports and shared configuration for the sentence-transformer WSD pipeline
from sentence_transformers import SentenceTransformer
from simple_wsd import process_senses_with_simple_wsd
from data_loader import load_sense_repo_by_round
from config import ANNOTATION_CHUNKS, DATA_DIR, OUTPUT_DIR, TESLA_SWD_MODEL
from writers import CustomWebAnnoTSVWriter, IncetprionWebAnnoTSVWriter

# Origin tag used when writing outputs
ORIGIN_WSD = "tesla_wsd"

# Annotation round to use (1 = old/first round, 2 = current/second round)
ROUND = 1

In [3]:
# 1) Instantiate the sentence-transformer model
wsd_model = SentenceTransformer(TESLA_SWD_MODEL)
print(f"Loaded model: {TESLA_SWD_MODEL}")

No sentence-transformers model found with name te-sla/TeslaXLM. Creating a new one with mean pooling.
Some weights of XLMRobertaModel were not initialized from the model checkpoint at te-sla/TeslaXLM and are newly initialized: ['embeddings.word_embeddings.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded model: te-sla/TeslaXLM


In [16]:
# 2) Load sense repository and select corpus chunk
senses_df = load_sense_repo_by_round(round=ROUND)
print(f"Loaded sense repo for round {ROUND}: {len(senses_df)} senses")

from webanno_spacy_converter.parsers.tsv_parser_v3 import WebAnnoLEXISParser

chunks = [(b, e, (DATA_DIR / fname)) for (b, e, fname) in ANNOTATION_CHUNKS]
print("Available chunks (index, begin, end, file):",
      [(i, b, e, p.name) for i, (b, e, p) in enumerate(chunks)])

chunk_idx = 4  # Select chunk index to process
chunk_begin, chunk_end, selected_path = chunks[chunk_idx]
print(f"Using chunk #{chunk_idx}: {selected_path.name} -> ({chunk_begin}, {chunk_end})")

parser = WebAnnoLEXISParser(selected_path)
sentences = parser.parse()

begin, end = chunk_begin, chunk_end

test = False
if test:
    tb, te = 242, 248
    sentences = sentences[tb:te]
    begin, end = tb + 1, te

Loaded sense repo for round 1: 12800 senses
Available chunks (index, begin, end, file): [(0, 1, 500, 'sr-elexis-WSD_0001_0500.tsv'), (1, 501, 1000, 'sr-elexis-WSD_0501_1000.tsv'), (2, 1001, 1500, 'sr-elexis-WSD_1001_1500.tsv'), (3, 1501, 2000, 'sr-elexis-WSD_1501_2000.tsv'), (4, 2001, 2024, 'sr-elexis-WSD_2001_2024.tsv')]
Using chunk #4: sr-elexis-WSD_2001_2024.tsv -> (2001, 2024)


In [17]:
# 3) Annotate sentences with cosine-similarity WSD
sentences = process_senses_with_simple_wsd(
    sentences,
    senses_df,
    model=wsd_model,
    origin_model=ORIGIN_WSD,
    progress=True,
)

Processed 24/24 sentences (simple_wsd).

In [18]:
# 4) Persist annotated corpus
ROUND_SUFFIX = f"_round{ROUND}"
writer = CustomWebAnnoTSVWriter(sentences)
writer.save(OUTPUT_DIR / f"LexiSense_{begin:04d}_{end:04d}_{ORIGIN_WSD}{ROUND_SUFFIX}.tsv")

incept_writer = IncetprionWebAnnoTSVWriter(sentences)
incept_writer.save(OUTPUT_DIR / f"LexiSense_Inception_{begin:04d}_{end:04d}_{ORIGIN_WSD}{ROUND_SUFFIX}.tsv")

# SimpleWSD_sense.ipynb

This notebook mirrors `ChatGPT_sense.ipynb` but relies on the local cosine-similarity pipeline (`process_senses_with_simple_wsd`) powered by sentence-transformer embeddings. Adjust `MODEL_NAME`, `chunk_idx`, or the `test` slice to process different corpora.